# Rotten Fruit Detector — YOLO26 (custom training container)

Fine-tunes **YOLO26** (Ultralytics, PyTorch) via a custom SageMaker training container
built from `docker/Dockerfile` (needed because it also exports to TensorFlow after
training). YOLO26 isn't in SageMaker JumpStart's built-in model zoo, so this bypasses
JumpStart entirely and runs a bring-your-own-container training job with a generic
`Estimator`. The image is built via AWS CodeBuild, triggered from this notebook — no
local Docker required.

Serving does **not** reuse that custom image: `ultralytics` has no compiled/GPU
build-time dependencies, so `deploy()` runs on AWS's stock managed PyTorch inference
container instead, extended the standard way with `serving/inference.py` +
`serving/requirements.txt` packaged into the model artifact.

Flow: build & push the training container (CodeBuild) → download a Roboflow **YOLOv8**
(Ultralytics) format export → upload it to S3 as-is (no conversion needed) → `.fit()`
launches an ephemeral GPU training instance running the image's `train` entrypoint →
deploy an endpoint on the stock inference container → predict.

Expected `dataset.zip` (Roboflow **YOLOv8** export — Ultralytics' format, works for
YOLO26 too):

```text
train/images/*.jpg + train/labels/*.txt
valid/images/*.jpg + valid/labels/*.txt
test/images/*.jpg  + test/labels/*.txt
data.yaml
```

Run on any small kernel (e.g. `ml.t3.medium`) — the heavy GPU work happens on the
separate training instance, not this notebook.

In [ ]:
# 1. Setup: SageMaker session, role, bucket
%pip install -q -r ../requirements.txt

import sys
import sagemaker
from pathlib import Path

session = sagemaker.Session()
role = sagemaker.get_execution_role()
bucket = session.default_bucket()
PROJECT_DIR = Path.cwd().parent
sys.path.insert(0, str(PROJECT_DIR / "src"))  # so `from train import train` etc. resolve
print("Role:", role)
print("Bucket:", bucket)

## One-time setup: let your Studio execution role build images

The next cell uses `sagemaker-studio-image-build`, which builds `docker/Dockerfile` via
AWS CodeBuild and pushes the result to ECR -- no local Docker needed. Your Studio
execution role (`role` above) needs these permissions attached once, before the first
run (ask whoever manages your AWS account's IAM if you don't have console access):

- `codebuild:CreateProject`, `codebuild:StartBuild`, `codebuild:BatchGetBuilds`, `codebuild:DeleteProject`
- `ecr:CreateRepository`, `ecr:InitiateLayerUpload`, `ecr:UploadLayerPart`, `ecr:CompleteLayerUpload`, `ecr:PutImage`, `ecr:BatchGetImage`
- `s3:GetObject`, `s3:PutObject`, `s3:DeleteObject` scoped to `arn:aws:s3:::sagemaker-*`
- `logs:CreateLogGroup`, `logs:CreateLogStream`, `logs:GetLogEvents`
- `iam:PassRole` for this same role (CodeBuild assumes it to push to ECR)

See the [sagemaker-studio-image-build README](https://github.com/aws-samples/sagemaker-studio-image-build-cli) for the exact policy JSON.

In [ ]:
# 2. Build the training image via CodeBuild and push it to ECR (no local Docker)
%pip install -q sagemaker-studio-image-build

REPOSITORY = "rotten-fruit-yolo26"

build_output = !cd {PROJECT_DIR / "docker"} && sm-docker build . --repository {REPOSITORY}:latest
print("\n".join(build_output))

IMAGE_URI = next(line.split("Image URI: ")[1] for line in build_output if "Image URI: " in line)
print("\nImage URI:", IMAGE_URI)

In [ ]:
# 3. Get the raw dataset from S3 and unzip it
import zipfile
import boto3

S3_DATA_ZIP = "s3://YOUR_BUCKET/datasets/dataset.zip"  # your zipped Roboflow YOLOv8-format export

b, key = S3_DATA_ZIP.replace("s3://", "").split("/", 1)
zip_path = PROJECT_DIR / "dataset.zip"
boto3.client("s3").download_file(b, key, str(zip_path))

EXTRACT_DIR = PROJECT_DIR / "dataset"
with zipfile.ZipFile(zip_path) as archive:
    archive.extractall(EXTRACT_DIR)

def find_dataset_root(base):
    base = Path(base)
    for path in [base, *(p for p in base.rglob("*") if p.is_dir())]:
        if (path / "data.yaml").exists():
            return path
    raise FileNotFoundError(f"No 'data.yaml' found under {base}")

DATA_DIR = find_dataset_root(EXTRACT_DIR)
print("Dataset root:", DATA_DIR)

In [ ]:
# 4. Upload the dataset root (Roboflow's YOLO export, as-is) to S3 (trailing slash required)
training_s3_uri = session.upload_data(
    path=str(DATA_DIR), bucket=bucket, key_prefix="rotten-fruit/yolo-input"
)
if not training_s3_uri.endswith("/"):
    training_s3_uri += "/"
print("Training data:", training_s3_uri)

In [ ]:
# 5. Fine-tune YOLO26 on an ephemeral GPU instance, using the image built in step 2.
#    ml.g5.2xlarge (A10G 24GB GPU, 32GB RAM). Needs its 'for training job usage' quota >= 1.
from train import train

estimator = train(
    training_s3_uri=training_s3_uri,
    role=role,
    image_uri=IMAGE_URI,
    output_path=f"s3://{bucket}/rotten-fruit/output",
    instance_type="ml.g5.2xlarge",
    epochs=75,
    batch_size=16,
    learning_rate=0.001,
    optimizer="SGD",
)

In [ ]:
# 6. Deploy an endpoint and run a prediction, then clean up
from train import deploy
from predict import predict_image

predictor = deploy(estimator, instance_type="ml.m5.xlarge")

test_image = sorted((DATA_DIR / "test" / "images").glob("*.jpg"))[0]
for det in predict_image(predictor, test_image)[:10]:
    print(det)

# IMPORTANT: delete the endpoint so it stops incurring charges.
predictor.delete_endpoint()